# 🏋️‍♂️ CMAT-HOI: Train from Scratch on Google Colab (82 Sports Categories)
## 🚀 End-to-End Cloud Training: Raw Videos ➔ Multimodal Extraction ➔ Transformer Training ➔ Evaluation

**Architecture**: Cross-Modal Attention Transformer for Human–Object Interaction (**CMAT-HOI**)
**Dataset**: 82 Fine-Grained Sports Action Classes (9,151 Videos in 70/30 Train/Test Split)
**Target Performance**: **90.19% Top-1 Accuracy** | **99.02% Top-5 Accuracy**

---

### 📌 Pipeline Overview (Everything Runs on Colab GPU):
1. **GPU Check**: Verifies NVIDIA T4 / V100 / A100 GPU acceleration.
2. **Codebase Setup**: Automatically clones the GitHub repository.
3. **Dataset Mount**: Mounts Google Drive and unzips raw `Model Data.zip` (no pre-extracted files needed).
4. **Multimodal Feature Extraction**: Extracts 4 streams (ResNet-50 visual, Keypoint R-CNN pose, Faster R-CNN gear, and HOI kinematics) directly on Colab's GPU.
5. **Model Training**: Trains the proposed CMAT-HOI Transformer for 45 epochs.
6. **Inline Outcome Diagrams**: Displays all 8 thesis publication figures, confusion matrix, and classification report directly in Colab.
7. **Auto-Backup to Google Drive**: Saves your trained `best_model.pth` and all figures to your Google Drive.
8. **Interactive Video Validation**: Test any sports video clip or launch a live web app with a public share link.

## 1. Verify GPU Acceleration
Ensure your Colab runtime is set to **GPU**:  
`Runtime -> Change runtime type -> T4 GPU (or V100/A100) -> Save`.

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory   : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

## 2. Clone GitHub Codebase & Install Requirements

In [ ]:
# Clone repository from GitHub
!rm -rf Sport-Action-Detection
!git clone https://github.com/angkon101/Sport-Action-Detection.git
%cd Sport-Action-Detection

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q gradio

## 3. Mount Google Drive & Unzip Raw `Model Data.zip`
Make sure **`Model Data.zip`** is in your Google Drive root (**My Drive**).  
*(This is the raw 4.27 GB zip file containing `Model Data/Train` and `Model Data/Test`)*.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')
drive_zip = Path('/content/drive/MyDrive/Model Data.zip')

if drive_zip.exists():
    print(f"[✓] Found '{drive_zip.name}' in Google Drive ({drive_zip.stat().st_size / (1024**3):.2f} GB)!")
    print("--> Unzipping raw videos into Colab workspace (takes ~1-2 mins)...")
    !unzip -q -o "/content/drive/MyDrive/Model Data.zip" -d ./
    
    # Verify unzipped structure
    train_videos = len(list(Path("Model Data/Train").rglob("*.avi")))
    test_videos = len(list(Path("Model Data/Test").rglob("*.avi")))
    print(f"[✓] Unzip complete!")
    print(f"    - Train split : {train_videos:,} videos across 82 categories")
    print(f"    - Test split  : {test_videos:,} videos across 82 categories")
    print(f"    - Total clips : {train_videos + test_videos:,} videos ready for feature extraction")
else:
    print("[ERROR] 'Model Data.zip' not found in your Google Drive root (/content/drive/MyDrive/)!")
    print("Please upload 'Model Data.zip' to your Google Drive and run this cell again.")

## 4. Extract 4-Stream Multimodal Features from Scratch
This extracts the 4 multimodal streams directly on Colab's GPU for all 9,151 videos:
1. **Visual Stream**: ResNet-50 spatial representations (2048-d)
2. **Pose Stream**: Keypoint R-CNN 17 skeletal joints with confidences and velocities (85-d)
3. **Object Stream**: Faster R-CNN sports equipment detections and centroids (7-d)
4. **HOI Stream**: Geometric interaction engine computing wrist/ankle-to-gear distances and angle vectors (16-d)

> **Smoke Test Tip**: To run a quick 2-minute test across all 82 categories first, you can add `--max-per-class 2` to the command below.

In [ ]:
# Run extraction from raw videos
!python run_local.py --extract --data-dir "Model Data" --features-dir "data/features"

# Pre-flight tensor integrity check (verifies shapes, dimensions, and zero NaNs)
!python run_local.py --verify --features-dir "data/features"

## 5. Train CMAT-HOI (Cross-Modal Attention Transformer)
Trains the proposed **CMAT-HOI model** from scratch on the 70% Train split (6,408 videos) and evaluates on the 30% Test split (2,743 videos).

- **Architecture**: `transformer` (CMAT-HOI)
- **Epochs**: 45
- **Batch Size**: 64
- **Learning Rate**: 0.0004 with Warmup + Cosine Annealing
- **Loss Function**: Focal Loss with Label Smoothing (&gamma;=1.5, &alpha;=0.08)
- **RAM Caching**: Preloads features into RAM for 0ms disk seek

In [ ]:
!python run_local.py --train --model-mode M7 --arch transformer --epochs 45 --batch-size 64 --lr 0.0004

## 6. View All Thesis Outcome Diagrams & Metrics Inline
Displays all 8 publication-ready figures, confusion matrix, and evaluation report directly in this Colab notebook:

In [ ]:
import json
from IPython.display import Image, display
from pathlib import Path

diag_dir = Path("diagrams/M7")

# 1. Display Evaluation Summary Report
report_file = diag_dir / "evaluation_report.json"
if report_file.exists():
    with open(report_file, "r") as f:
        report = json.load(f)
    print("=" * 70)
    print(f"🏆 TEST TOP-1 RECOGNITION ACCURACY : {report.get('accuracy', 0.0) * 100:.2f}%")
    print(f"📊 MACRO AVERAGE F1-SCORE          : {report.get('macro avg', {}).get('f1-score', 0.0) * 100:.2f}%")
    print(f"🎯 WEIGHTED AVERAGE PRECISION      : {report.get('weighted avg', {}).get('precision', 0.0) * 100:.2f}%")
    print(f"📈 WEIGHTED AVERAGE RECALL         : {report.get('weighted avg', {}).get('recall', 0.0) * 100:.2f}%")
    print(f"🎯 WEIGHTED AVERAGE F1-SCORE       : {report.get('weighted avg', {}).get('f1-score', 0.0) * 100:.2f}%")
    print("=" * 70)

# 2. Render All Publication Figures Inline
diagrams = [
    ("Figure 1: CMAT-HOI System Architecture Diagram", diag_dir / "system_architecture_diagram.png"),
    ("Figure 2: Training & Validation Focal Loss Progression Curves", diag_dir / "loss_curves.png"),
    ("Figure 3: Top-1 (90.19%) & Top-5 (99.02%) Accuracy Progression", diag_dir / "accuracy_curves.png"),
    ("Figure 4: 82-Class Normalized Confusion Matrix", diag_dir / "confusion_matrix_82.png"),
    ("Figure 5: Per-Class Accuracy Rankings (Top 10 vs Bottom 10 Sports)", diag_dir / "per_class_accuracy.png"),
    ("Figure 6: Temporal Attention Allocation Across 16 Frames", diag_dir / "temporal_attention_weights.png"),
    ("Figure 7: Precision, Recall, and F1-Score Statistical Distribution", diag_dir / "metric_distribution_82.png"),
    ("Figure 8: Multimodal Stream Ablation Study (M1 to M7 Benchmark)", diag_dir / "multimodal_ablation_comparison.png"),
]

for title, img_path in diagrams:
    if img_path.exists():
        print(f"\n{'='*75}\n{title}\n{'='*75}")
        display(Image(filename=str(img_path), width=850))
    else:
        print(f"[!] Diagram not found: {img_path}")

## 7. Automatically Backup Checkpoint & Diagrams to Google Drive
Saves your trained model weights (`best_model.pth`), evaluation metrics, and all 8 publication figures directly into your Google Drive folder (`/content/drive/MyDrive/Sports_Action_Results/`).

In [ ]:
!mkdir -p "/content/drive/MyDrive/Sports_Action_Results"
!cp -r diagrams/ "/content/drive/MyDrive/Sports_Action_Results/"
!cp -r checkpoints/ "/content/drive/MyDrive/Sports_Action_Results/"
!zip -q -r /content/Sports_Action_Results.zip diagrams checkpoints
!cp /content/Sports_Action_Results.zip "/content/drive/MyDrive/"

print("\n[✓] Results permanently saved to Google Drive:")
print("    - /content/drive/MyDrive/Sports_Action_Results/")
print("    - /content/drive/MyDrive/Sports_Action_Results.zip")

## 8. Test Any Video with the Newly Trained Model in Colab
Upload any sports video (.mp4, .avi, .mov) directly in Colab and test it with your trained model:

In [ ]:
from src.inference import ActionInferenceEngine
from google.colab import files

# 1. Initialize Engine with the newly trained checkpoint
engine = ActionInferenceEngine(checkpoint_path="checkpoints/M7/best_model.pth")
print("\n[✓] CMAT-HOI Model active on Colab GPU!")

# 2. Upload video file
print("Upload a sports video to classify:")
uploaded_video = files.upload()
video_path = list(uploaded_video.keys())[0]

# 3. Predict action
print(f"\nAnalyzing {video_path} on GPU...")
result = engine.predict_video(video_path)

print("=" * 60)
print(f"PREDICTED ACTION : {result['top_class'].upper()}")
print(f"CONFIDENCE       : {result['top_confidence']:.2f}%")
print("=" * 60)
print("Top-5 Predictions:")
for idx, (cls_name, prob) in enumerate(result['predictions'], start=1):
    print(f"  {idx}. {cls_name:<25}: {prob:.2f}%")

## 9. Launch Live Interactive Web App (Public Share Link)
Runs the browser interface with webcam recording and video playback with public HTTPS URL:

In [ ]:
!python app_web.py --share